# Exploratory analysis – BRFSS 2015 Diabetes Health Indicators
Run from the `notebooks/` folder. Keep this short: it supports checkpoint 1 (EDA notes) and the decision log.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import sys; sys.path.insert(0, '..')
import core
df = pd.read_csv('../data/diabetes_binary_health_indicators_BRFSS2015.csv')
print(df.shape)
df.dtypes.value_counts()

## 1. Quality checks: missing values, duplicates, ranges

In [ ]:
print('missing values:', int(df.isna().sum().sum()))
n_dup = int(df.duplicated().sum())
print('duplicate rows:', n_dup, f'({n_dup/len(df):.1%})')
df.describe().T[['min','max','mean']].round(2)

## 2. Target distribution (imbalanced)
Raw vs after removing duplicates.

In [ ]:
print('raw positive rate:', round(df[core.TARGET].mean(), 4))
clean = df.drop_duplicates()
print('after de-duplication:', round(clean[core.TARGET].mean(), 4), '| rows:', len(clean))
clean[core.TARGET].value_counts().plot(kind='bar', title='Target counts (0 = no, 1 = diabetes/prediabetes)'); plt.show()

## 3. Positive rate by key variables
Looking for strong signals and checking group sizes before the fairness analysis.

In [ ]:
for col in ['HighBP', 'HighChol', 'GenHlth', 'Age', 'Income', 'Education']:
    t = clean.groupby(col)[core.TARGET].agg(['mean', 'size']).round(3)
    print(col); print(t.T.to_string()); print()

In [ ]:
clean.groupby(pd.cut(clean.BMI, [0, 18.5, 25, 30, 35, 100]))[core.TARGET].agg(['mean', 'size']).round(3)

## 4. Group sizes used for the equity analysis

In [ ]:
print(core.income_group(clean.Income).value_counts()); print(core.edu_group(clean.Education).value_counts())

## 5. Leakage check
No column is derived from the target. Variables like `HighBP`/`HighChol`/`GenHlth` are self-reported answers a user can give at home. Variables that depend on a lab test are not used in the short questionnaire.

In [ ]:
clean.corr()[core.TARGET].drop(core.TARGET).sort_values(ascending=False).round(3)

## 6. Notes for the write-up
- Write 4-6 sentences here: size, duplicates, imbalance, strongest signals, any surprises.
- Then see `train.py` for the full pipeline and `reports/` for results.